# 01 · EDA & Reference Suitability Audit — PySpark DataFrame
Kết quả này hỗ trợ quyết định Normal/Super, **không tự coi XSum/XL-Sum = Super hoặc VietNews = Normal**.   
**Phân biệt phép đo:**
- `source_units`, `target_units` = số đoạn ngăn bằng khoảng trắng, không phải token của mô hình.
- `reference_sentence_proxy` = số câu *ước tính từ dấu câu/newline*, cần human audit.
- `lexical_overlap_proxy` và `lead_overlap_proxy` chỉ đánh giá độ trùng từ, không chứng minh factuality hoặc semantic coverage.
- `source_hash` dùng chuẩn hóa khoảng trắng + chữ thường để phát hiện trùng chính xác; chưa nhận diện near-duplicate.

In [1]:
from pathlib import Path
import os, sys, json, hashlib, csv, platform
from datetime import datetime, timezone
from pprint import pprint

HERE = Path.cwd().resolve()
ROOT = HERE.parent if HERE.name.lower() == "notebooks" else HERE
if not (ROOT / "notebooks").is_dir():
    raise RuntimeError(f"Hãy mở VS Code ở thư mục dự án chứa notebooks/. CWD hiện tại: {HERE}")
RAW = ROOT / "data" / "raw"
SILVER = ROOT / "data" / "silver"
GOLD = ROOT / "data" / "gold"
REPORTS = ROOT / "reports"
for p in (RAW, REPORTS): p.mkdir(parents=True, exist_ok=True)
print("PROJECT ROOT:", ROOT)
print("PYTHON:", sys.executable)
print("LOGICAL CORES:", os.cpu_count())

PROJECT ROOT: D:\DuLieuCuaHuu\HK1_20262027\BigData\CK_BIG
PYTHON: c:\Users\Lenovo\AppData\Local\Programs\Python\Python313\python.exe
LOGICAL CORES: 16


In [2]:
SPLITS = ("train", "validation", "test")
DATASETS = {
    "cnn_dailymail": {
        "repo": "abisee/cnn_dailymail", "prefix": "3.0.0", "revision": None,
        "id": "id", "source": "article", "target": "highlights", "language": "en",
        "expected": {"train": 287113, "validation": 13368, "test": 11490},
    },
    "xsum": {
        "repo": "EdinburghNLP/xsum", "prefix": "data", "revision": None,
        "id": "id", "source": "document", "target": "summary", "language": "en",
        "expected": {"train": 204045, "validation": 11332, "test": 11334},
    },
    "vietnews": {
        "repo": "nam194/vietnews", "prefix": "data", "revision": None,
        "id": "guid", "source": "article", "target": "abstract", "language": "vi",
        "expected": {"train": 99134, "validation": 22184, "test": 22498},
    },
    "xlsum_vi": {
        "repo": "csebuetnlp/xlsum", "prefix": "vietnamese",
        "revision": "fcbaaeaec2ec660c6164ea74e8e7371b66883670",
        "id": "id", "source": "text", "target": "summary", "language": "vi",
        "expected": {"train": 32111, "validation": 4013, "test": 4013},
    },
}

def json_write(path, obj):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(json.dumps(obj,ensure_ascii=False,indent=2),encoding="utf-8")

def csv_write(path, rows, columns=None):
    rows=list(rows)
    cols=columns or (list(rows[0]) if rows else [])
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    with path.open("w",encoding="utf-8-sig",newline="") as f:
        w=csv.DictWriter(f,fieldnames=cols,extrasaction="ignore")
        w.writeheader(); w.writerows(rows)

def file_sha256(path):
    h=hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda:f.read(8*1024*1024),b""):h.update(block)
    return h.hexdigest()

def load_manifest(name):
    path=RAW/name/"source_manifest.json"
    if not path.exists():
        raise FileNotFoundError(f"Chưa có {path}. Hãy chạy notebook 00 trước.")
    m=json.loads(path.read_text(encoding="utf-8"))
    if m.get("dataset") != name or not m.get("valid"):
        raise ValueError(f"Manifest chưa đạt/không khớp: {path}")
    return m

def split_files(name, split):
    m=load_manifest(name)
    if split not in m["splits"]: raise ValueError(f"Thiếu split {split}: {name}")
    fs=[RAW/name/item["path"] for item in m["splits"][split]["files"]]
    if not fs or not all(p.is_file() for p in fs):
        raise FileNotFoundError(f"Thiếu Parquet {name}/{split}. Chạy notebook 00.")
    return [str(p) for p in fs]

In [3]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"

from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql import types as T

SPARK_CORES = min(12, os.cpu_count() or 4)  
SPARK_DRIVER_MEMORY = "4g"                 # Giảm còn 2g nếu RAM máy thấp
SHUFFLE_PARTITIONS = max(16, SPARK_CORES * 3)

spark = (SparkSession.builder
    .appName("HybriNews-Local")
    .master(f"local[{SPARK_CORES}]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.python.worker.reuse", "false")
    .config("spark.sql.shuffle.partitions", str(SHUFFLE_PARTITIONS))
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .config("spark.driver.memory", SPARK_DRIVER_MEMORY)
    .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version, "| master:", spark.sparkContext.master,
      "| worker reuse:", spark.sparkContext.getConf().get("spark.python.worker.reuse"))
assert spark.sparkContext.getConf().get("spark.python.worker.reuse").lower()=="false", "Restart kernel rồi chạy lại cell Spark."

Spark 4.1.2 | master: local[12] | worker reuse: false


In [4]:
# Đọc một dataset
def raw_dataset(name):
    cfg=DATASETS[name]
    dfs=[]
    for split in SPLITS:
        df=spark.read.parquet(*split_files(name,split))
        required=(cfg["id"],cfg["source"],cfg["target"])
        missing=set(required)-set(df.columns)
        if missing: raise ValueError(f"{name}/{split}: thiếu các cột {sorted(missing)}")
        dfs.append(df.select(
            F.lit(name).alias("dataset"),
            F.lit(cfg["language"]).alias("language"),
            F.lit(split).alias("split"),
            F.col(cfg["id"]).cast("string").alias("article_id"),
            F.col(cfg["source"]).cast("string").alias("source_text"),
            F.col(cfg["target"]).cast("string").alias("target_text"),
        ))
    data=dfs[0].unionByName(dfs[1]).unionByName(dfs[2])
    def norm(col):
        return F.trim(F.regexp_replace(F.coalesce(col,F.lit("")),r"\s+"," "))
    data=data.withColumn("source_norm",norm(F.col("source_text")))
    data=data.withColumn("target_norm",norm(F.col("target_text")))
    data=data.withColumn("source_hash",F.sha2(F.lower(F.col("source_norm")),256))
    data=data.withColumn("target_hash",F.sha2(F.col("target_norm"),256))
    # Không phụ thuộc thứ tự row sau shuffle.
    data=data.withColumn("record_id",F.sha2(F.concat_ws("|", "dataset", "split",
         F.coalesce(F.col("article_id"),F.lit("")), "source_hash", "target_hash"),256))
    return data

## Các hàm thống kê DataFrame

In [5]:
import pandas as pd
import matplotlib.pyplot as plt

def nonblank(col):
    return F.length(F.trim(F.coalesce(col,F.lit(""))))>0
def count_ws(col):
    s=F.trim(F.coalesce(col,F.lit("")))
    return F.when(F.length(s)==0,F.lit(0)).otherwise(F.size(F.split(s,r"\s+")))

def enriched(df):
    text=F.trim(F.coalesce(F.col("target_text"),F.lit("")))
    return (df
        .withColumn("source_units",count_ws(F.col("source_text")))
        .withColumn("target_units",count_ws(F.col("target_text")))
        .withColumn("sentence_proxy",F.when(F.length(text)==0,F.lit(0))
            .otherwise(F.size(F.split(text,r"(?<=[.!?。！？])\s+|\n+"))))
        .withColumn("compression_ratio",F.when(F.col("source_units")>0,
            F.col("target_units")/F.col("source_units")).otherwise(F.lit(None)))
    )
def quantiles(df,col):
    # Các phân vị APPROXIMATE, không phải percentile chính xác tuyệt đối.
    return df.select(F.percentile_approx(F.col(col),[0.5,0.9,0.95,0.99],10000).alias("p")).first()["p"]
def audit_one(name):
    cfg=DATASETS[name]
    folder=REPORTS/"eda"/name
    folder.mkdir(parents=True,exist_ok=True)
    df=enriched(raw_dataset(name))
    counts=[r.asDict() for r in df.groupBy("split").agg(
        F.count("*").alias("n"),
        F.sum((~nonblank(F.col("source_text"))).cast("long")).alias("empty_source"),
        F.sum((~nonblank(F.col("target_text"))).cast("long")).alias("empty_target"),
        F.avg((F.col("sentence_proxy")==1).cast("double")).alias("fraction_sentence1_proxy"),
        F.avg("compression_ratio").alias("mean_compression"),
    ).collect()]
    stats=[]
    for split in SPLITS:
        row=next((r for r in counts if r["split"]==split),None)
        if row is None: raise ValueError(f"{name} thiếu split {split}")
        part=df.filter(F.col("split")==split)
        a=quantiles(part,"source_units")
        b=quantiles(part,"target_units")
        c=quantiles(part,"sentence_proxy")
        cr=quantiles(part.filter(F.col("compression_ratio").isNotNull()),"compression_ratio") or [None]*4
        stats.append({"dataset":name,"split":split,"n":row["n"],
          "empty_source":row["empty_source"],"empty_target":row["empty_target"],
          "source_p50":a[0],"source_p90":a[1],"source_p95":a[2],"source_p99":a[3],
          "target_p50":b[0],"target_p90":b[1],"target_p95":b[2],"target_p99":b[3],
          "sentence_p50_proxy":c[0],"sentence_p95_proxy":c[2],
          "one_sentence_ratio_proxy":row["fraction_sentence1_proxy"],
          "compression_p50":cr[0],"compression_p95":cr[2],
          "mean_compression":row["mean_compression"]})
    csv_write(folder/"reference_statistics.csv",stats)
    n_all=sum(x["n"] for x in stats)
    expected=sum(cfg["expected"].values())
    if n_all!=expected: raise ValueError(f"{name}: Spark đọc {n_all} bản ghi, expected={expected}")

    valid=df.filter(F.col("source_units")>0)
    dup=(valid.groupBy("source_hash").agg(F.count("*").alias("copies"),
         F.countDistinct("target_hash").alias("target_variants"),
         F.countDistinct("split").alias("split_variants"))
         .filter(F.col("copies")>1))
    duplicate_summary=dup.agg(
       F.count("*").alias("groups"),
       F.coalesce(F.sum(F.col("copies")-1),F.lit(0)).alias("excess_records"),
       F.coalesce(F.sum((F.col("target_variants")>1).cast("long")),F.lit(0)).alias("reference_variant_groups"),
       F.coalesce(F.sum((F.col("split_variants")>1).cast("long")),F.lit(0)).alias("cross_split_groups")
       ).first().asDict()
    duplicate_summary["dataset"]=name
    csv_write(folder/"duplicate_summary.csv",[duplicate_summary])
    # Chỉ xuất số nhóm trùng có giới hạn; KHÔNG dùng collect toàn corpus.
    dup.orderBy(F.desc("copies"),"source_hash").limit(300).toPandas().to_csv(
        folder/"duplicate_groups_top300.csv",index=False,encoding="utf-8-sig")
    leaks=(dup.filter(F.col("split_variants")>1).select("source_hash")
         .join(valid.select("source_hash","record_id","split","article_id","target_hash"),"source_hash","inner"))
    leaks.orderBy("source_hash","split").limit(500).toPandas().to_csv(
        folder/"cross_split_examples_max500.csv",index=False,encoding="utf-8-sig")

    # Lấy mẫu có quy tắc xác định, tối đa 30 dòng mỗi split. Tạo phiếu HUMAN REVIEW.
    sample_w=Window.partitionBy("split").orderBy(F.xxhash64("record_id"),"record_id")
    sample=(df.withColumn("_rank",F.row_number().over(sample_w))
            .filter(F.col("_rank")<=30).drop("_rank"))
    annotations=sample.select("dataset","split","article_id","record_id",
        "source_text","target_text","source_units","target_units","sentence_proxy").toPandas()
    for col in ("reference_factuality_human","normal_suitability_human",
                "super_suitability_human","lead_bias_human","reviewer","notes"):
        annotations[col]=""
    annotations.to_csv(REPORTS/"audit"/f"{name}_human_review.csv",index=False,encoding="utf-8-sig")

    # Chỉ tính lexical proxies trên một mẫu để giữ tốc độ, không tính toàn bộ 722k dòng.
    tokens=lambda col: F.array_distinct(F.split(
        F.trim(F.regexp_replace(F.lower(F.coalesce(col,F.lit(""))),r"[^\p{L}\p{N}]+"," ")),r"\s+"))
    proxy=(sample.filter((F.col("source_units")>0)&(F.col("target_units")>0))
       .withColumn("source_tokens",tokens(F.col("source_norm")))
       .withColumn("target_tokens",tokens(F.col("target_norm")))
       .withColumn("lead_tokens",F.slice(F.col("source_tokens"),1,200))
       .withColumn("lexical_overlap_proxy",F.size(F.array_intersect("target_tokens","source_tokens")) /
                   F.greatest(F.size("target_tokens"),F.lit(1)))
       .withColumn("lead_overlap_proxy",F.size(F.array_intersect("target_tokens","lead_tokens")) /
                   F.greatest(F.size("target_tokens"),F.lit(1))))
    lex=[r.asDict() for r in proxy.groupBy("split").agg(F.count("*").alias("sample_n"),
         F.avg("lexical_overlap_proxy").alias("mean_lexical_overlap_proxy"),
         F.avg("lead_overlap_proxy").alias("mean_lead_overlap_proxy")).collect()]
    for x in lex:x["dataset"]=name
    csv_write(folder/"sampled_lexical_proxies.csv",lex)

    plot_df=pd.DataFrame(stats)
    fig,axes=plt.subplots(1,2,figsize=(10,3.5))
    axes[0].bar(plot_df["split"],plot_df["source_p50"])
    axes[0].set_title(f"{name} · Source P50 (whitespace units)")
    axes[1].bar(plot_df["split"],plot_df["target_p50"])
    axes[1].set_title(f"{name} · Reference P50")
    fig.tight_layout();fig.savefig(folder/"length_medians.png",dpi=160);plt.close(fig)
    print(f"PASS {name}: {n_all:,} rows | duplicate groups:",duplicate_summary["groups"])
    display(plot_df)
    display(pd.DataFrame(lex))
    return {"dataset":name,"rows":n_all,**duplicate_summary}

## 01-A · CNN/DailyMail

In [7]:
cnn_audit = audit_one("cnn_dailymail")

PASS cnn_dailymail: 311,971 rows | duplicate groups: 3103


,dataset,split,n,empty_source,empty_target,source_p50,source_p90,source_p95,source_p99,target_p50,target_p90,target_p95,target_p99,sentence_p50_proxy,sentence_p95_proxy,one_sentence_ratio_proxy,compression_p50,compression_p95,mean_compression
0,cnn_dailymail,train,287113,0,0,632,1165,1362,1673,48,77,90,119,4,6,0.015994,0.078224,0.188341,0.090759
1,cnn_dailymail,validation,13368,0,0,606,1166,1369,1667,53,86,98,127,4,7,0.004189,0.091603,0.213198,0.104520
2,cnn_dailymail,test,11490,0,0,613,1185,1395,1686,51,81,94,124,4,6,0.002785,0.085928,0.205212,0.099940


,split,sample_n,mean_lexical_overlap_proxy,mean_lead_overlap_proxy,dataset
0,test,30,0.824709,0.764169,cnn_dailymail
1,validation,30,0.871502,0.805706,cnn_dailymail
2,train,30,0.855120,0.789753,cnn_dailymail


## 01-B · XSum

In [8]:
xsum_audit = audit_one("xsum")

PASS xsum: 226,711 rows | duplicate groups: 56


,dataset,split,n,empty_source,empty_target,source_p50,source_p90,source_p95,source_p99,target_p50,target_p90,target_p95,target_p99,sentence_p50_proxy,sentence_p95_proxy,one_sentence_ratio_proxy,compression_p50,compression_p95,mean_compression
0,xsum,train,204045,28,0,295,754,926,1329,21,27,29,36,1,1,0.999142,0.070552,0.287356,0.100258
1,xsum,validation,11332,5,0,292,751,920,1320,21,27,29,36,1,1,0.999647,0.071429,0.283333,0.100049
2,xsum,test,11334,1,0,295,749,926,1362,21,27,29,37,1,1,0.999206,0.070707,0.278481,0.099341


,split,sample_n,mean_lexical_overlap_proxy,mean_lead_overlap_proxy,dataset
0,test,30,0.619598,0.579110,xsum
1,train,30,0.605759,0.585024,xsum
2,validation,30,0.626177,0.597842,xsum


## 01-C · VietNews

In [9]:
vietnews_audit = audit_one("vietnews")

PASS vietnews: 143,816 rows | duplicate groups: 0


,dataset,split,n,empty_source,empty_target,source_p50,source_p90,source_p95,source_p99,target_p50,target_p90,target_p95,target_p99,sentence_p50_proxy,sentence_p95_proxy,one_sentence_ratio_proxy,compression_p50,compression_p95,mean_compression
0,vietnews,train,99134,0,0,356,731,838,1029,27,40,44,54,1,2,0.799564,0.074699,0.162879,0.083059
1,vietnews,validation,22184,0,0,360,735,843,1036,27,40,44,54,1,2,0.792508,0.074627,0.163793,0.082974
2,vietnews,test,22498,0,0,360,736,845,1024,27,40,44,55,1,2,0.791537,0.074906,0.164336,0.083461


,split,sample_n,mean_lexical_overlap_proxy,mean_lead_overlap_proxy,dataset
0,test,30,0.857424,0.813083,vietnews
1,train,30,0.852272,0.813499,vietnews
2,validation,30,0.857866,0.811574,vietnews


## 01-D · XL-Sum Vietnamese

In [10]:
xlsum_audit = audit_one("xlsum_vi")

PASS xlsum_vi: 40,137 rows | duplicate groups: 3


,dataset,split,n,empty_source,empty_target,source_p50,source_p90,source_p95,source_p99,target_p50,target_p90,target_p95,target_p99,sentence_p50_proxy,sentence_p95_proxy,one_sentence_ratio_proxy,compression_p50,compression_p95,mean_compression
0,xlsum_vi,train,32111,0,5,665,2005,2580,5749,32,49,57,73,1,2,0.918533,0.049451,0.156757,0.062811
1,xlsum_vi,validation,4013,0,0,497,937,1044,1161,30,42,46,53,1,1,0.968104,0.062049,0.143460,0.070755
2,xlsum_vi,test,4013,0,0,495,940,1048,1157,31,42,46,54,1,1,0.968602,0.062500,0.142276,0.070595


,split,sample_n,mean_lexical_overlap_proxy,mean_lead_overlap_proxy,dataset
0,test,30,0.805986,0.762376,xlsum_vi
1,train,30,0.812713,0.669634,xlsum_vi
2,validation,30,0.766535,0.725121,xlsum_vi


## Kiểm tra trùng nguồn xuyên dataset (trong cùng ngôn ngữ)

In [11]:
# Dùng source_hash tính từ source_norm, không phân biệt chữ hoa/thường.
# Các trùng *gần giống* cần phương pháp riêng, chưa được chứng minh là hết leakage.
parts=[]
for name in DATASETS:
    parts.append(raw_dataset(name).filter(F.length("source_norm")>0)
        .select("dataset","language","split","source_hash"))
union=parts[0]
for part in parts[1:]: union=union.unionByName(part)
cross=(union.groupBy("language","source_hash")
       .agg(F.countDistinct("dataset").alias("datasets"),
            F.countDistinct("split").alias("splits"),F.count("*").alias("rows"))
       .filter(F.col("datasets")>1))
cross_summary=cross.agg(F.count("*").alias("groups"),
    F.coalesce(F.sum((F.col("splits")>1).cast("long")),F.lit(0)).alias("cross_split_groups")).first().asDict()
json_write(REPORTS/"eda"/"cross_dataset_overlap_summary.json",cross_summary)
cross.orderBy(F.desc("rows")).limit(500).toPandas().to_csv(
    REPORTS/"eda"/"cross_dataset_overlap_top500.csv",index=False,encoding="utf-8-sig")
print("Cross-dataset exact overlap:",cross_summary)

Cross-dataset exact overlap: {'groups': 0, 'cross_split_groups': 0}


## Tổng hợp và biểu mẫu quyết định nhiệm vụ — CHƯA TỰ ĐỘNG DUYỆT

In [12]:
all_audits=[cnn_audit,xsum_audit,vietnews_audit,xlsum_audit]
csv_write(REPORTS/"eda"/"overall_summary.csv",all_audits)
print("Tổng các bản ghi:",sum(r["rows"] for r in all_audits))

# Chỉ tạo template để nghiên cứu viên kiểm tra, KHÔNG tạo file approved sẵn.
decision_path=REPORTS/"audit"/"task_decisions.template.json"
if not decision_path.exists():
    decision={"approved":False,"approved_by":"","approved_date":"",
      "human_audit_completed":False,"near_duplicate_reviewed":False,
      "cross_dataset_leakage_reviewed":False,
      "datasets":{name:{"task":None,"approved":False,"rationale":""} for name in DATASETS}}
    json_write(decision_path,decision)
print("Mẫu quyết định:",decision_path)

Tổng các bản ghi: 722635
Mẫu quyết định: D:\DuLieuCuaHuu\HK1_20262027\BigData\CK_BIG\reports\audit\task_decisions.template.json
